## Time Series Analysis

### 1. Setup and Data Loading

First, we'll install necessary libraries, load the Apple (AAPL) stock data, and perform initial preprocessing.

In [ ]:
# Install necessary libraries
!pip install yfinance pmdarima tensorflow xgboost scikit-learn statsmodels

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

# For statistical models
from statsmodels.tsa.api import ExponentialSmoothing, SimpleExpSmoothing, Holt
from pmdarima import auto_arima

# For machine learning models
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# For deep learning models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Set display options for pandas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Download historical stock data for Apple (AAPL)
# We'll use a relatively long period to capture various market conditions
start_date = '2010-01-01'
end_date = '2023-12-31'

df = yf.download('AAPL', start=start_date, end=end_date)

# Display the first few rows
display(df.head())

### 2. Data Preprocessing

We will focus on the 'Close' price and ensure our data is properly indexed for time series analysis. We'll also visualize the series to understand its patterns.

In [ ]:
# Select the 'Close' price and set the index as datetime
ts_data = df.loc[:, ('Close', 'AAPL')].copy() # Explicitly select the 'Close' price for 'AAPL' as a Series
ts_data.index = pd.to_datetime(ts_data.index)

# Set frequency for the time series index, crucial for statsmodels forecasting
ts_data = ts_data.asfreq('B')

# Check for missing values
print(f"Missing values before handling: {ts_data.isnull().sum()}")

# Fill any missing values (e.g., using forward fill or interpolation)
ts_data = ts_data.fillna(method='ffill') # Forward fill is common for time series

print(f"Missing values after handling: {ts_data.isnull().sum()}")

# Visualize the time series
plt.figure(figsize=(14, 7))
plt.plot(ts_data)
plt.title('AAPL Stock Close Price Over Time')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.show()

### 3. Splitting Data into Training and Testing Sets

To evaluate our models effectively, we split the data into a training set (e.g., 80%) and a testing set (e.g., 20%). The models will be trained on the training data and evaluated on the unseen testing data.

In [ ]:
# Define the split point (e.g., 80% for training, 20% for testing)
train_size = int(len(ts_data) * 0.8)
train_data, test_data = ts_data[0:train_size], ts_data[train_size:len(ts_data)]

print(f"Training data points: {len(train_data)}")
print(f"Testing data points: {len(test_data)}")

# Plot the split
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Training Data')
plt.plot(test_data.index, test_data, label='Testing Data', color='orange')
plt.title('AAPL Stock Price: Training and Testing Split')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

### 4. Forecasting Methods

Now we will implement and evaluate each of the requested forecasting methods. For each method, we will:
1.  Train the model on `train_data`.
2.  Generate predictions for the `test_data` period.
3.  Calculate evaluation metrics (MAE, RMSE).
4.  Visualize the forecast against the actual test data.

#### 4.1 Naive Method

The Naive method predicts the next value to be the same as the last observed value from the training set. It's a simple baseline.

In [ ]:
def evaluate_forecast(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{model_name} MAE: {mae:.2f}")
    print(f"{model_name} RMSE: {rmse:.2f}")
    return mae, rmse

# Naive Forecast
naive_predictions = np.full_like(test_data, train_data.iloc[-1])

mae_naive, rmse_naive = evaluate_forecast(test_data, naive_predictions, 'Naive Method')

# Plot Naive Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='orange')
plt.plot(test_data.index, naive_predictions, label='Naive Forecast', color='green', linestyle='--')
plt.title('Naive Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.2 Average Method

This method predicts the next value to be the average of all past observations in the training set.

In [ ]:
# Average Forecast
average_predictions = np.full_like(test_data, train_data.mean())

mae_average, rmse_average = evaluate_forecast(test_data, average_predictions, 'Average Method')

# Plot Average Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='orange')
plt.plot(test_data.index, average_predictions, label='Average Forecast', color='purple', linestyle='--')
plt.title('Average Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.3 Drift Method

The Drift method is a variation of the Naive method that allows the forecasts to increase or decrease over time, where the amount of change over time (called the 'drift') is set to be the average change seen in the historical data.

In [ ]:
# Drift Forecast
def drift_forecast(train_data, horizon):
    if len(train_data) < 2:
        return np.full(horizon, train_data.iloc[-1]) # Fallback to naive if not enough data

    last_value = train_data.iloc[-1]
    # Calculate the average change per step (drift)
    drift = (train_data.iloc[-1] - train_data.iloc[0]) / (len(train_data) - 1)

    predictions = [last_value + drift * (i + 1) for i in range(horizon)]
    return np.array(predictions)

drift_predictions = drift_forecast(train_data, len(test_data))

mae_drift, rmse_drift = evaluate_forecast(test_data, drift_predictions, 'Drift Method')

# Plot Drift Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='orange')
plt.plot(test_data.index, drift_predictions, label='Drift Forecast', color='brown', linestyle='--')
plt.title('Drift Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.4 ARIMA (AutoRegressive Integrated Moving Average)

ARIMA models are a class of statistical models for analyzing and forecasting time series data. We will use `pmdarima.auto_arima` to automatically find the optimal ARIMA parameters (p, d, q).

In [ ]:
# ARIMA Forecast
# Use auto_arima to find the best ARIMA model parameters
# seasonal=False because stock prices usually don't have a fixed seasonal pattern
print("Searching for optimal ARIMA model parameters...")
arima_model = auto_arima(train_data, seasonal=False, suppress_warnings=True, stepwise=True,
                         trace=True, error_action='ignore', max_p=7, max_d=5, max_q=7)
print(arima_model.summary())

# Make predictions
arima_forecast_output = arima_model.predict(n_periods=len(test_data))

# Ensure predictions are a 1D array/Series before assigning index
if isinstance(arima_forecast_output, pd.Series) and arima_forecast_output.index.is_integer():
    arima_predictions_values = arima_forecast_output.values
else:
    arima_predictions_values = arima_forecast_output

arima_predictions = pd.Series(arima_predictions_values, index=test_data.index)

# Evaluate forecast, ensuring test_data is a 1D Series
mae_arima, rmse_arima = evaluate_forecast(test_data.squeeze(), arima_predictions, 'ARIMA Method')

# Plot ARIMA Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='orange')
plt.plot(arima_predictions.index, arima_predictions, label='ARIMA Forecast', color='red', linestyle='--')
plt.title('ARIMA Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.4.2 ARIMA with Log Transform

Applying a logarithmic transformation can help stabilize the variance of the time series and make it more stationary, which often improves the performance of ARIMA models. We will apply `np.log` to the data before training and `np.exp` to the predictions before evaluating them.

In [ ]:
# ARIMA Forecast with Log Transform

# Apply log transformation to the data
train_data_log = np.log(train_data)
test_data_log = np.log(test_data)

print("Searching for optimal ARIMA model parameters on log-transformed data...")
arima_model_log = auto_arima(train_data_log, seasonal=False, suppress_warnings=True, stepwise=True,
                         trace=True, error_action='ignore', max_p=7, max_d=5, max_q=7)
print(arima_model_log.summary())

# Make predictions on the log scale
arima_forecast_output_log = arima_model_log.predict(n_periods=len(test_data_log))

# Inverse transform the predictions back to the original scale
arima_predictions_log = np.exp(arima_forecast_output_log)

# Ensure predictions are a 1D array/Series before assigning index
if isinstance(arima_predictions_log, pd.Series) and arima_predictions_log.index.is_integer():
    arima_predictions_values_log = arima_predictions_log.values
else:
    arima_predictions_values_log = arima_predictions_log

arima_predictions_log = pd.Series(arima_predictions_values_log, index=test_data.index)

# Evaluate forecast, ensuring test_data is a 1D Series
mae_arima_log, rmse_arima_log = evaluate_forecast(test_data.squeeze(), arima_predictions_log, 'ARIMA (Log Transformed) Method')

# Plot ARIMA Forecast with Log Transform
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='orange')
plt.plot(arima_predictions_log.index, arima_predictions_log, label='ARIMA (Log Transformed) Forecast', color='darkred', linestyle='--')
plt.title('ARIMA (Log Transformed) Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.5 Exponential Smoothing (Holt's Linear Trend)

Exponential smoothing methods are a family of forecasting techniques. For data with a trend, Holt's linear trend method is often used. It extends simple exponential smoothing to allow for forecasting data with a trend.

In [ ]:
# Exponential Smoothing (Holt's Linear Trend) Forecast
# Fit Holt's Linear Trend Model
holt_model = Holt(train_data, initialization_method="estimated").fit()
holt_predictions = holt_model.forecast(len(test_data))
holt_predictions = pd.Series(holt_predictions, index=test_data.index)

mae_holt, rmse_holt = evaluate_forecast(test_data, holt_predictions, 'Holt Method')

# Plot Holt Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(test_data.index, test_data, label='Actual Test Data', color='orange')
plt.plot(holt_predictions.index, holt_predictions, label='Holt Forecast', color='cyan', linestyle='--')
plt.title('Holt Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

### 4.6 Feature Engineering and Data Scaling for ML/DL Models

For machine learning and deep learning models like Random Forests, XGBoost, and LSTMs, we need to transform the time series data into a supervised learning problem. This typically involves creating lagged features (past values of the target variable) and scaling the data.

In [ ]:
# Create lagged features
def create_features(data, lag_steps=1):
    df_temp = pd.DataFrame(data)
    columns = [df_temp.shift(i) for i in range(1, lag_steps + 1)]
    columns.append(df_temp)
    df_temp = pd.concat(columns, axis=1)
    df_temp.columns = [f'Lag_{i}' for i in range(1, lag_steps + 1)] + ['Target']
    df_temp.dropna(inplace=True)
    return df_temp

lag_steps = 5 # Number of previous days to use as features

train_features = create_features(train_data, lag_steps)
test_features = create_features(test_data, lag_steps)

X_train, y_train = train_features.iloc[:, :-1], train_features.iloc[:, -1]
X_test, y_test = test_features.iloc[:, :-1], test_features.iloc[:, -1]

# Scale the data using MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep track of the original test target for evaluation
y_test_original_index = y_test.index

print("Shape of X_train_scaled:", X_train_scaled.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test_scaled:", X_test_scaled.shape)
print("Shape of y_test:", y_test.shape)

# Create a dummy scaler for y_train and y_test if they are not already scaled
y_scaler = MinMaxScaler(feature_range=(0, 1))
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

#### 4.7 Random Forest Regressor

Random Forest is an ensemble learning method that can be used for forecasting. It works by constructing multiple decision trees during training and outputting the mean prediction of the individual trees.

In [ ]:
# Random Forest Forecast
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train_scaled.ravel())

rf_predictions_scaled = rf_model.predict(X_test_scaled)
rf_predictions = y_scaler.inverse_transform(rf_predictions_scaled.reshape(-1, 1))
rf_predictions = pd.Series(rf_predictions.flatten(), index=y_test_original_index)

mae_rf, rmse_rf = evaluate_forecast(y_test, rf_predictions, 'Random Forest Method')

# Plot Random Forest Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(y_test.index, y_test, label='Actual Test Data', color='orange')
plt.plot(rf_predictions.index, rf_predictions, label='Random Forest Forecast', color='navy', linestyle='--')
plt.title('Random Forest Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.8 XGBoost Regressor

XGBoost (eXtreme Gradient Boosting) is an optimized distributed gradient boosting library designed to be highly efficient, flexible, and portable. It implements machine learning algorithms under the Gradient Boosting framework.

In [ ]:
# XGBoost Forecast
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train_scaled.ravel())

xgb_predictions_scaled = xgb_model.predict(X_test_scaled)
xgb_predictions = y_scaler.inverse_transform(xgb_predictions_scaled.reshape(-1, 1))
xgb_predictions = pd.Series(xgb_predictions.flatten(), index=y_test_original_index)

mae_xgb, rmse_xgb = evaluate_forecast(y_test, xgb_predictions, 'XGBoost Method')

# Plot XGBoost Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(y_test.index, y_test, label='Actual Test Data', color='orange')
plt.plot(xgb_predictions.index, xgb_predictions, label='XGBoost Forecast', color='darkgreen', linestyle='--')
plt.title('XGBoost Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

#### 4.9 LSTM (Long Short-Term Memory)

LSTMs are a special kind of Recurrent Neural Network (RNN) capable of learning long-term dependencies. They are well-suited for time series forecasting. We'll prepare our scaled data for LSTM by reshaping it into a 3D format (samples, timesteps, features).

In [ ]:
# LSTM Forecast
# Reshape input to be [samples, time steps, features]
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_test_lstm = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

print("Shape of X_train_lstm:", X_train_lstm.shape)
print("Shape of X_test_lstm:", X_test_lstm.shape)

# Build the LSTM model
lstm_model = Sequential()
lstm_model.add(LSTM(units=50, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
lstm_model.add(Dense(units=1))
lstm_model.compile(optimizer='adam', loss='mean_squared_error')

# Train the LSTM model
history = lstm_model.fit(X_train_lstm, y_train_scaled, epochs=50, batch_size=32, verbose=0, shuffle=False)

# Make predictions
lstm_predictions_scaled = lstm_model.predict(X_test_lstm)
lstm_predictions = y_scaler.inverse_transform(lstm_predictions_scaled.reshape(-1, 1))
lstm_predictions = pd.Series(lstm_predictions.flatten(), index=y_test_original_index)

mae_lstm, rmse_lstm = evaluate_forecast(y_test, lstm_predictions, 'LSTM Method')

# Plot LSTM Forecast
plt.figure(figsize=(14, 7))
plt.plot(train_data.index, train_data, label='Train Data')
plt.plot(y_test.index, y_test, label='Actual Test Data', color='orange')
plt.plot(lstm_predictions.index, lstm_predictions, label='LSTM Forecast', color='darkred', linestyle='--')
plt.title('LSTM Method Forecast vs Actuals')
plt.xlabel('Date')
plt.ylabel('Close Price (USD)')
plt.legend()
plt.show()

### 5. Summary of Model Performance

Let's compare the performance of all the forecasting models used.

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive', 'Average', 'Drift', 'ARIMA', 'Holt', 'Random Forest', 'XGBoost', 'LSTM'],
    'MAE': [mae_naive, mae_average, mae_drift, mae_arima, mae_holt, mae_rf, mae_xgb, mae_lstm],
    'RMSE': [rmse_naive, rmse_average, rmse_drift, rmse_arima, rmse_holt, rmse_rf, rmse_xgb, rmse_lstm]
})

results = results.sort_values(by='RMSE')
display(results)

In [ ]:
results = pd.DataFrame({
    'Model': ['Naive', 'Average', 'Drift', 'ARIMA', 'ARIMA (Log Transformed)', 'Holt', 'Random Forest', 'XGBoost', 'LSTM'],
    'MAE': [mae_naive, mae_average, mae_drift, mae_arima, mae_arima_log, mae_holt, mae_rf, mae_xgb, mae_lstm],
    'RMSE': [rmse_naive, rmse_average, rmse_drift, rmse_arima, rmse_arima_log, rmse_holt, rmse_rf, rmse_xgb, rmse_lstm]
})

results = results.sort_values(by='RMSE')
display(results)